# Laboratorio 2 · Contexto y ejecución

Grupo de **tres estudiantes**. Completar nombres antes de entregar. Se usa el CSV PaySim adaptado del Laboratorio 1. Los archivos de `references/` son material docente; sus resultados no se usan como si fueran datos del grupo.

In [1]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from engine import fmt, pformat, load_dataset, run_full_analysis, descriptive_stats
from narrative import sections
r = json.loads((ROOT / 'results/analysis_summary.json').read_text(encoding='utf-8'))
h = pd.read_csv(ROOT / 'results/hourly_series.csv')
pd.options.display.float_format = lambda x: f'{x:.4f}'
def tabla(rows):
    frame = pd.DataFrame(rows)
    for col in frame:
        if col.endswith('_p') or col == 'p_value': frame[col] = frame[col].map(pformat)
    display(frame)
def apartado(i):
    title, texts = sections(r)[i-1]
    display(Markdown('## ' + title))
    for text in texts: display(Markdown(text))
def figura(name):
    display(Image(filename=str(ROOT / 'results/figures' / name), width=850))


In [2]:
apartado(1)
tabla(r['variable_dictionary'])
display(r['data_quality'])

## 1. Contexto, población y calidad

El archivo verificado contiene 6,362,620 transacciones y 7 variables originales. La unidad original es una transacción. La población de referencia son las operaciones del sistema simulado PaySim bajo sus reglas generadoras; el archivo es la muestra disponible, no una muestra probabilística de todos los usuarios bancarios reales. Si se delimita la población al archivo, las descriptivas son censales.

La naturaleza analítica elegida es serie de tiempo. Se construye y(t) = número de filas con step = t para pronosticar el volumen de operaciones por hora. Al agregar cambia la unidad de observación a hora. step es tiempo discreto; monto es la única continua original. Los identificadores numéricos y de cuentas se clasifican como nominales. No hay variable ordinal original ni etiqueta de fraude en estas siete columnas.

Se observaron 0 valores faltantes, 0 identificadores duplicados, 0 montos negativos y 16 montos cero. Las comprobaciones de prefijo de destino dieron 0 inconsistencias. Los ceros se conservan y se dejan señalados para revisar su origen. No se eliminan operaciones solo por ser grandes.

La serie tiene 743 steps entre 1 y 743. Se valida continuidad antes de calcular ACF; una hora ausente no se interpreta automáticamente como cero. La fase step % 24 es reproducible, pero no se conoce una fecha ni un reloj civil. Tampoco se deducen días de la semana reales.

Método: descriptivas, tablas categóricas, Levene, ANOVA y Kruskal usan el archivo completo; Pearson/Spearman/Kendall a nivel transacción usan muestra aleatoria simple sin reemplazo de hasta 200000 filas (semilla 42). Las correlaciones horarias usan todas las horas. Shapiro usa hasta 5000 observaciones por grupo. Nivel de referencia alpha = 0.0500; no se redondea antes de decidir.

,variable,tipo,definicion
0,id_transaccion,Nominal (identificador),Clave entera; no usar como predictor cuantitat...
1,step,Cuantitativa discreta,Índice horario de la simulación; fase inicial ...
2,tipo_transaccion,Nominal,"CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER."
3,monto,Cuantitativa continua,Unidades monetarias del simulador; no se asume...
4,cuenta_origen,Nominal,Identificador de cuenta emisora.
5,cuenta_destino,Nominal,Identificador de cuenta receptora.
6,tipo_cuenta_destino,Nominal,CLIENTE o COMERCIO.
7,transacciones_hora,Discreta derivada,Conteo de filas por step; unidad hora.
8,monto_promedio / monto_mediano / monto_total,Continuas derivadas,"Media, mediana y suma de monto por hora."
9,fase_24 / franja_horaria,Cíclica discreta / nominal derivada,"step % 24 y bloques 00-05, 06-11, 12-17, 18-23."


{'missing_by_column': {'id_transaccion': 0,
  'step': 0,
  'tipo_transaccion': 0,
  'monto': 0,
  'cuenta_origen': 0,
  'cuenta_destino': 0,
  'tipo_cuenta_destino': 0},
 'empty_text_by_column': {'tipo_transaccion': 0,
  'cuenta_origen': 0,
  'cuenta_destino': 0,
  'tipo_cuenta_destino': 0},
 'duplicate_id_transaccion': 0,
 'negative_monto': 0,
 'zero_monto': 16,
 'nonfinite_monto': 0,
 'invalid_step': 0,
 'invalid_tipo_transaccion': [],
 'unknown_destination_type': 0,
 'destination_prefix_inconsistency_known_types': 0}

## Verificación de origen

La huella SHA-256 identifica el CSV utilizado. Si el archivo está disponible se puede recalcular desde este notebook o desde `scripts/run_analysis.py`. La apertura normal presenta las salidas reales guardadas.

In [3]:
display(r['metadata'])
assert r['metadata']['columns'] == 7
assert int(h.transacciones_hora.sum()) == r['metadata']['rows']
print('Reconciliación de filas y conteos horarios: correcta')

{'source_file': 'transacciones.csv',
 'source_sha256': '262ae36b53af5b4a7798a1429d389710ff6926f2e81f2d6842e1b3626765f684',
 'source_bytes': 385653971,
 'rows': 6362620,
 'columns': 7,
 'column_names': ['id_transaccion',
  'step',
  'tipo_transaccion',
  'monto',
  'cuenta_origen',
  'cuenta_destino',
  'tipo_cuenta_destino'],
 'sample_n_for_correlations': 200000,
 'sample_seed': 42,
 'alpha': 0.05,
 'rounding_digits': 4,
 'python': '3.14.7',
 'package_versions': {'pandas': '3.1.0',
  'numpy': '2.5.4',
  'scipy': '1.18.1',
  'statsmodels': '0.15.0',
  'scikit-learn': '1.9.1',
  'matplotlib': '3.11.2',
  'streamlit': '1.66.1'}}

Reconciliación de filas y conteos horarios: correcta


In [4]:
# Cambiar a True para rehacer todo con el CSV original.
RECALCULAR = False
CSV = Path(os.environ.get('LAB2_CSV', str(ROOT / 'data/transacciones.csv')))
if RECALCULAR:
    df = load_dataset(CSV)
    r = run_full_analysis(CSV, ROOT / 'results', df=df)
    from engine import build_hourly
    from plots import generate_figures
    generate_figures(df, build_hourly(df), ROOT / 'results/figures')
    print('Análisis actualizado; regenerar PDF con scripts/create_report.py')
else:
    print('Mostrando resultados reales guardados. Para recalcular: RECALCULAR=True y ruta CSV.')

Mostrando resultados reales guardados. Para recalcular: RECALCULAR=True y ruta CSV.


## Reparto sugerido

Integrante 1: contexto, calidad y descriptivas. Integrante 2: asociaciones y pruebas. Integrante 3: serie temporal, dashboard y ejecución. Los tres validan la interpretación final. No se suministraron nombres.

La guía exige siete apartados analíticos y un octavo entregable: notebooks, proyecto VSCode y Streamlit. La cobertura detallada está en `reports/Matriz_cumplimiento.md`.